# Modul 06 · Notebook 00 — Ekosistem NVIDIA & GPU

Selamat datang di **modul terakhir**! Di Modul 05 kamu sudah membangun **RAG bot** yang pintar. Di modul ini kita jawab dua pertanyaan besar:

1. **Bagaimana menjalankannya dengan efisien** di hardware NVIDIA? → nb00–03
2. **Bagaimana membuatnya aman & layak dipercaya** untuk 10.000 user sungguhan? → nb04–07 (*Trustworthy AI* + *guardrails*)

### Peta perjalanan 8 notebook
| # | Notebook | Inti |
|---|----------|------|
| 00 | Ekosistem NVIDIA & GPU | GPU/CUDA, benchmark FP16 vs FP32 |
| 01 | Optimasi Inferensi | quantization 4-bit |
| 02 | NVIDIA NIM | generator cloud (tanpa GPU) |
| 03 | RAG di NIM | RAG di atas stack NVIDIA |
| 04 | Trustworthy AI & 5 Rail | 4 pilar + guardrails pertama |
| 05 | Keamanan & Privasi | jailbreak/toxicity + PII masking |
| 06 | Grounding & Topik | anti-halusinasi + topical rail |
| 07 | Capstone Deploy | service ber-guardrails penuh |

### Istilah dasar (untuk pemula)
- **GPU** (*Graphics Processing Unit*): chip dengan ribuan core kecil yang bekerja **paralel** — ideal untuk perkalian matriks di balik AI.
- **CUDA**: platform NVIDIA yang membuat GPU bisa diprogram untuk komputasi umum, bukan cuma grafis.
- **FP32 / FP16** (*floating point* 32-bit / 16-bit): presisi angka. FP16 memakai **separuh** memori dan biasanya lebih cepat di GPU modern.
- **Inference**: tahap **memakai** model yang sudah dilatih untuk menghasilkan output (lawan dari *training*).

> ⚙️ **Runtime:** pastikan Colab memakai **GPU T4** — menu *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Instal dependensi (pinned untuk reproducibility)
!pip -q install "transformers>=4.53,<5" "accelerate>=0.30" 

In [ ]:
# Cek GPU — Cara 1: CLI bawaan NVIDIA
!nvidia-smi

In [ ]:
# Cek GPU — Cara 2: lewat PyTorch
import torch
print("CUDA tersedia :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("Nama GPU      :", torch.cuda.get_device_name(0))
    print("Compute cap.  :", torch.cuda.get_device_capability(0))
    print("Memori total  :", round(p.total_memory / 1e9, 1), "GB")
    print("Memori dipakai:", round(torch.cuda.memory_allocated(0) / 1e9, 2), "GB")
else:
    print("GPU tidak aktif - aktifkan T4 via Runtime -> Change runtime type.")

## Kenapa GPU jauh lebih cepat untuk AI?

CPU punya sedikit core yang sangat cepat — bagus untuk tugas berurutan. GPU punya **ribuan core** yang lebih lambat satu-satu, tapi bekerja **bersamaan**. Inference LLM = jutaan perkalian matriks yang bisa dikerjakan paralel → cocok untuk GPU.

Mari **buktikan**: kita jalankan model yang **sama** dalam dua presisi (FP32 vs FP16) lalu bandingkan **memori** dan **kecepatan**.

In [ ]:
# Helper benchmark: muat model dalam satu presisi, ukur memori + kecepatan generate
import torch, time, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)

def run_dtype(dtype, label, prompt="Jelaskan apa itu GPU dalam satu kalimat sederhana.", n=5):
    torch.cuda.empty_cache(); gc.collect()
    model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map="auto")
    mem_gb = torch.cuda.memory_allocated() / 1e9                       # memori bobot model
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt").to(model.device)
    attn = torch.ones_like(ids)
    model.generate(ids, attention_mask=attn, max_new_tokens=8,
                   do_sample=False, pad_token_id=tok.eos_token_id)     # warmup
    t0 = time.time()
    for _ in range(n):
        out = model.generate(ids, attention_mask=attn, max_new_tokens=64,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    dt = (time.time() - t0) / n
    txt = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    return {"label": label, "mem_gb": round(mem_gb, 2), "sec": round(dt, 3), "text": txt.strip()}

In [ ]:
# Jalankan FP32 (presisi penuh)
r32 = run_dtype(torch.float32, "FP32")
r32

In [ ]:
# Jalankan FP16 (separuh presisi)
r16 = run_dtype(torch.float16, "FP16")
r16

In [ ]:
# Bandingkan: memori, kecepatan, dan OUTPUT (cek trade-off akurasi)
import pandas as pd
df = pd.DataFrame([r32, r16])[["label", "mem_gb", "sec"]]
df["speedup_x"] = (r32["sec"] / df["sec"]).round(2)
print(df.to_string(index=False))
print("\n--- Output FP32 ---\n", r32["text"])
print("\n--- Output FP16 ---\n", r16["text"])

## Yang kita pelajari

- **FP16 ≈ separuh memori** dari FP32, biasanya **lebih cepat**, dengan output **nyaris identik** untuk model ini.
- Itulah sebabnya hampir semua inference LLM modern memakai FP16 (atau lebih hemat lagi).
- Di **nb01** kita melangkah lebih jauh: **quantization 4-bit**, yang memangkas memori jauh lebih banyak lagi.

## 🧪 Latihan

1. Ganti `NAME` menjadi `"Qwen/Qwen2.5-0.5B-Instruct"` lalu jalankan ulang — apa yang berubah pada memori & kecepatan?
2. Tambah satu presisi lagi: `torch.bfloat16`. Apakah hasilnya berbeda dengan FP16 di T4? (Catatan: T4 mendukung FP16 dengan baik; `bfloat16` lebih cocok di GPU Ampere ke atas.)